# GIADA Task 17b — attribuzione causale dei rate Ca_HVA
Follow-up diagnostico del NO-GO 17a: formula, LUT completa, quattro singoli rate e coppia m. Nessuna soglia cambiata, nessuna autorizzazione Gate C. Leggere `experiments/task17b_rate_attribution_preregistration.md`.

In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys,zipfile,uuid,tempfile
from IPython.display import Javascript,display
WORK=Path('/kaggle/working')/f'giada_task17b_{uuid.uuid4().hex[:8]}'
REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net';WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_task17b_rate_attribution.py').is_file(),'Revisione GIADA senza Task 17b.'
print({'giada_revision':REVISION,'teacher_revision':'074c4666300a8ad246601dab179a97a6942f0f29'})

## Preflight NEURON
Il codice, il notebook e il protocollo devono provenire dallo stesso commit GIADA.

In [ ]:
try:
 import neuron
except ModuleNotFoundError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+',1)[0]=='8.2.7',f'NEURON {neuron.__version__} non corrisponde a 8.2.7.'
nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl')
assert Path(nrnivmodl).is_file() and shutil.which('gcc'),'Compilatore NEURON/gcc assente.'
simulation_dir=TEACHER/'L5PC_NEURON_simulation'
if not list(simulation_dir.rglob('libnrnmech.so')):subprocess.run([nrnivmodl,'mods'],cwd=simulation_dir,check=True)
assert list(simulation_dir.rglob('libnrnmech.so')),'Meccanismi canonici non compilati.'
print({'neuron':neuron.__version__,'canonical_mechanisms':'ready'})

## Input
Serve solo l'artefatto Task 15c come Dataset Kaggle; non serve l'HDF5 targeted. In caso di mount insolito, impostare `GIADA_TASK15C_ARTIFACT` al file ZIP o alla cartella estratta.

In [ ]:
sys.path.insert(0,str(REPO))
from src.giada_teacher.roadmap_embedded_frozen_confirmation import verified_15c
INPUT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK15C_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT.is_dir():
 candidates += list(INPUT.rglob('*.zip'))
 candidates += [p.parent for p in INPUT.rglob('final_report.json') if 'task15c' in str(p).lower() or 'interface_bridge' in str(p).lower()]
TASK15C=None
for candidate in dict.fromkeys(candidates):
 if not candidate.exists():continue
 try:verified_15c(candidate);TASK15C=candidate.resolve();break
 except (ValueError,RuntimeError,FileNotFoundError,OSError,zipfile.BadZipFile):continue
assert TASK15C is not None,'Artefatto Task 15c esatto non trovato negli Input Kaggle.'
print({'task15c':str(TASK15C)})

## Matrice nativa appaiata
Otto bracci × tre seed × tre conduttanze sul protocollo NMDA, ~72 traiettorie. Il runner conserva report e log anche in caso di fallimento. I risultati sono solo diagnostici.

In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/f'giada_task17b_rate_attribution_{REVISION[:7]}_{uuid.uuid4().hex[:8]}'
command=[sys.executable,'-u',str(REPO/'scripts/run_roadmap_task17b.py'),'--repo',str(REPO),'--teacher',str(TEACHER),'--task15c',str(TASK15C),'--output',str(OUTPUT)]
completed=subprocess.run(command,check=False)
result_path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').is_file() else 'final_report.json')
if result_path.is_file():report=json.loads(result_path.read_text())
else:report={'valid':False,'decision':'SUPERVISOR_INTERRUPTED','error':f'exit={completed.returncode}; controllare process.log'}
display({key:report.get(key) for key in ('valid','decision','gate_c_authorized','full_lut_worst_gate_error','error_type','error')})
print({'output':str(OUTPUT),'returncode':completed.returncode,'next':'Scarica lo ZIP anche se il processo fallisce.'})

## ZIP diagnostico
Lo ZIP contiene metriche, tracciati del caso difficile, provenienza e log; non stampa array estesi nel notebook.

In [ ]:
EXPORT=Path(tempfile.mkdtemp(prefix='task17b_download_',dir=WORK))
for name in ('final_report.json','failure_report.json','process_status.json','process.log','last_phase.json','paired_metrics.json','diagnostic_traces.json'):
 source=OUTPUT/name
 if source.is_file():shutil.copy2(source,EXPORT/name)
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',EXPORT.parent,EXPORT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})